In [1]:
#implementing vad by using RMS energy

import librosa

In [18]:
def silence_detect(audio_path, threshold = 0.03):

    audio,sr = librosa.load(audio_path,sr = 16000)
    print(f"audio {len(audio)}")
    print(f"sr {sr}")


    #frame length = 25ms

    frame_length = int(0.025*16000)

    #Hop length = 10s -> to overlap frames to get more good results

    hop_length = int(0.01*16000)

    rms_energy = librosa.feature.rms(
        y = audio,
        frame_length = frame_length,
        hop_length = hop_length
    )[0]

    #convert rms values into time frames

    time_frames = librosa.frames_to_time(
        range(len(rms_energy)),
        sr=sr,
        hop_length=hop_length
    )

    silence_segments = []

    silence_start = None
    min_silence = 0.2

    for i, energy in enumerate(rms_energy):
        if energy < threshold:
            if silence_start is None:
                silence_start = time_frames[i]

        else:
            if silence_start is not None:
                  silence_end = time_frames[i]
                  if silence_end - silence_start >= min_silence:
                      silence_segments.append({
                    "start": float(round(silence_start, 2)),
                    "end": float(round(silence_end, 2))
                })

                  silence_start = None
    if silence_start is not None:
        duration = (len(audio) / sr) - silence_start

        if duration >= min_silence:
            silence_segments.append({
            "start": float(round(silence_start, 2)),
            "end": round(len(audio) / sr, 2)
        })

    return silence_segments




In [19]:
segments = silence_detect("/content/audio1.wav")

for segment in segments:
    print(segment)


audio 273760
sr 16000
{'start': 0.0, 'end': 0.84}
{'start': 1.58, 'end': 2.34}
{'start': 6.24, 'end': 6.45}
{'start': 8.85, 'end': 9.09}
{'start': 9.59, 'end': 9.95}
{'start': 11.41, 'end': 11.71}
{'start': 12.11, 'end': 14.72}
{'start': 15.0, 'end': 15.33}
{'start': 15.69, 'end': 17.11}
